In [33]:
!pip install scikit-learn
!pip install imbalanced-learn
!pip install joblib
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 MB 103.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.7/291.7 MB 97.0 MB/s eta 0:00:0000:0100:01


In [41]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.under_sampling import RandomUnderSampler, NearMiss, ClusterCentroids
from joblib import dump, load
import matplotlib.pyplot as plt
import seaborn as sns
import time

In [4]:
# Load Engineered data
df = pd.read_csv("FE_&_EDA.csv")

/var/tmp/ipykernel_11073/2349710039.py:2: DtypeWarning: Columns (54,56,57,67) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('FE_&_EDA.csv')


In [5]:
# feature selection
drop_cols = [
    "Year",
    "Month",
    "Day",
    "Time",
    "Merchant Name",
    "Merchant City",
    "Merchant State",
    "Zip",
    "Is Fraud?",
    "Current Age",
    "Retirement Age",
    "Birth Year",
    "Birth Month",
    "Gender",
    "Address",
    "Apartment",
    "City",
    "State",
    "Zipcode",
    "Latitude",
    "Longitude",
    "Per Capita Income - Zipcode",
    "Yearly Income - Person",
    "Card Number",
    "Expires",
    "CVV",
    "Acct Open Date",
    "Year PIN last Changed",
    "Card on Dark Web",
    "timestamp",
    "Date",
    "Acct Open Year",
    "low_fico_high_spend",
    "Low_FICO_High_DTI",
    "next_mcc",
    "time_diff",
    "amt_diff",
    "is_us_zip",
    "min_acc_open_date",
    "max_acc_open_date",
]

In [6]:
# Encoding categorical columns
categorical_cols = [
    "Use Chip",
    "Errors?",
    "Card Brand",
    "Card Type",
    "Has Chip",
    "Day_of_Week",
    "Transaction_type",  #'High_risk_state','High_risk_cities',
    "Age Group",  #'Is Retired',
    "Retirement Proximity",
    "Retirement Phase",
    "Zip Income Tier",
    "income_tier",
    "income_mismatch",  #'high_risk_transactions', ,'high_debt_ratio', #'high_debt_high_spend',
    "credit_util_bin",
    "fico_tier",  #'synthetic_risk', 'synthetic_risk_2', 'High_risk_MCC', 'mcc_changed', 'Rapid_mcc_changed', 'High_risk_merchant', 'zip_mismatch_flag'
]

In [7]:
# Defining X and Y
X = df.drop(columns=drop_cols, errors="ignore")
y = df["Is Fraud?"]

In [8]:
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

In [9]:
# scale numeric features
numeric_cols = [
    "Amount",
    "MCC",
    "Total Debt",
    "FICO Score",
    "Num Credit Cards",
    "Cards Issued",
    "Credit Limit",
    "Transaction Hour",
    "Years to retirement",
    "Years Since Retirement",
    "amount_income_ratio",
    "income_relative_to_zip",
    "debt_to_income",
    "credit_utilization",
    "account_tenure_years",
    "unique_mcc_count",
    "months_to_expiry",
]

While applying RobustScaler, I encountered a ValueError. Upon inspecting the numerical columns, I discovered that the "Income relative to zip" column contained infinite values. This occurred because the income per zip code was 0 for some zip codes—likely due to missing data for those zip codes, as the dataset is synthetic. To resolve this issue, I decided to replace the infinite values with 0.

In [10]:
X[numeric_cols].dtypes

Amount                    float64
MCC                         int64
Total Debt                float64
FICO Score                  int64
Num Credit Cards            int64
Cards Issued                int64
Credit Limit              float64
Transaction Hour            int64
Years to retirement       float64
Years Since Retirement    float64
amount_income_ratio       float64
income_relative_to_zip    float64
debt_to_income            float64
credit_utilization        float64
account_tenure_years      float64
unique_mcc_count            int64
months_to_expiry          float64
dtype: object

In [11]:
# Returns True/False for each column in numeric_cols based on whether it has inf or -inf values
X[numeric_cols].apply(lambda col: np.isinf(col).any())

Amount                    False
MCC                       False
Total Debt                False
FICO Score                False
Num Credit Cards          False
Cards Issued              False
Credit Limit              False
Transaction Hour          False
Years to retirement       False
Years Since Retirement    False
amount_income_ratio       False
income_relative_to_zip     True
debt_to_income            False
credit_utilization        False
account_tenure_years      False
unique_mcc_count          False
months_to_expiry          False
dtype: bool

In [12]:
X[numeric_cols].apply(lambda col: np.isinf(col).sum())

Amount                        0
MCC                           0
Total Debt                    0
FICO Score                    0
Num Credit Cards              0
Cards Issued                  0
Credit Limit                  0
Transaction Hour              0
Years to retirement           0
Years Since Retirement        0
amount_income_ratio           0
income_relative_to_zip    56257
debt_to_income                0
credit_utilization            0
account_tenure_years          0
unique_mcc_count              0
months_to_expiry              0
dtype: int64

In [13]:
X[numeric_cols] = X[numeric_cols].replace([np.inf, -np.inf], np.nan)  # Replace with NaN
X[numeric_cols] = X[numeric_cols].fillna(0)  # Then fill NaNs with 0 or another method

In [14]:
# Try a low threshold, like 0.0001
selector = VarianceThreshold(threshold=0.0001)
X_reduced = selector.fit_transform(X)

dropped_features = X.columns[~selector.get_support()]
print("Dropped low-variance features:", dropped_features.tolist())

Dropped low-variance features: ['High_risk_synthetic_user', 'Errors?_Bad CVV,Insufficient Balance', 'Errors?_Bad CVV,Technical Glitch', 'Errors?_Bad Card Number,Bad CVV', 'Errors?_Bad Card Number,Bad Expiration', 'Errors?_Bad Card Number,Insufficient Balance', 'Errors?_Bad Card Number,Technical Glitch', 'Errors?_Bad Expiration,Bad CVV', 'Errors?_Bad Expiration,Insufficient Balance', 'Errors?_Bad Expiration,Technical Glitch', 'Errors?_Bad PIN,Insufficient Balance', 'Errors?_Bad PIN,Technical Glitch', 'Errors?_Bad Zipcode', 'Errors?_Bad Zipcode,Insufficient Balance', 'Errors?_Bad Zipcode,Technical Glitch', 'Errors?_Insufficient Balance,Technical Glitch']


In [15]:
# Keep the columns with sufficient variance
X = X.loc[:, selector.get_support()]

In [16]:
scaler = RobustScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols].fillna(0))

In [17]:
# # Subsample a balanced or random portion of your data
# X_small = X.sample(n=500_000, random_state=42)
# y_small = y.loc[X_small.index]

In [45]:
# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42, stratify=y
)

In [19]:
# Baseline model - Logistic Regression
model_baseline = LogisticRegression(
    class_weight="balanced", max_iter=10000, random_state=42
)
model_baseline.fit(X_train, y_train)

# Predictions and evaluation
y_pred = model_baseline.predict(X_test)
y_pred_proba = model_baseline.predict_proba(X_test)[:, 1]

In [20]:
# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# ROC-AUC score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

In [21]:
# Save trained model
dump(model_baseline, "model_baseline.joblib")

In [22]:
# PCA (on original data)
pca = PCA(n_components=0.95, random_state=42)  # Retain 95% variance
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
print("\nPCA Components:", X_train_pca.shape[1])
print("Explained Variance Ratio:", sum(pca.explained_variance_ratio_))

# Logistic Regression with PCA
pca_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
pca_model.fit(X_train_pca, y_train)
y_pred_pca = pca_model.predict(X_test_pca)
y_pred_proba_pca = pca_model.predict_proba(X_test_pca)[:, 1]
print("\nPCA Logistic Regression - Classification Report:")
print(classification_report(y_test, y_pred_pca))
roc_auc_pca = roc_auc_score(y_test, y_pred_proba_pca)
print(f"PCA ROC-AUC: {roc_auc_pca:.4f}")

In [23]:
# Save trained model
dump(pca_model, "pca_model.joblib")

In [24]:
# ADASYN oversampling

adasyn = ADASYN(sampling_strategy="auto", random_state=42, n_neighbors=5)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)
print("\nADASYN Train Shape:", X_train_adasyn.shape)
print("Fraud in ADASYN Train:", y_train_adasyn.sum())

In [25]:
# Logistic Regression with ADASYN
adasyn_model = LogisticRegression(
    max_iter=1000, random_state=42
)  # No class_weight since balanced
adasyn_model.fit(X_train_adasyn, y_train_adasyn)

In [26]:
# ADASYN predictions
y_pred_adasyn = adasyn_model.predict(X_test)
y_pred_proba_adasyn = adasyn_model.predict_proba(X_test)[:, 1]

In [27]:
# ADASYN evaluation
print("\nADASYN Logistic Regression - Classification Report:")
print(classification_report(y_test, y_pred_adasyn))
roc_auc_adasyn = roc_auc_score(y_test, y_pred_proba_adasyn)
print(f"ADASYN ROC-AUC: {roc_auc_adasyn:.4f}")

In [28]:
# try modeling with different technics of undersampling


def evaluate_model(X_train, y_train, X_test, y_test, sampler, sampler_name):
    print(f"\n=== {sampler_name} ===")

    # Downsample
    X_resampled, y_resampled = sampler.fit_resample(X, y)

    # Fit Logistic Regression
    model = LogisticRegression(class_weight="balanced", max_iter=10000, random_state=42)

    start_time = time.time()
    model.fit(X_resampled, y_resampled)
    end_time = time.time()

    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # save the m
    dump(model, f"{sampler_name}.joblib")

    # Evaluation
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
    print(f"Training Time: {end_time - start_time:.2f} seconds")

In [29]:
# Run for RandomUnderSampler
evaluate_model(
    X_train,
    y_train,
    X_test,
    y_test,
    RandomUnderSampler(random_state=42),
    "Random UnderSampler",
)

# Run for NearMiss
evaluate_model(X_train, y_train, X_test, y_test, NearMiss(version=1), "NearMiss (v1)")

# Run for ClusterCentroids
evaluate_model(
    X_train,
    y_train,
    X_test,
    y_test,
    ClusterCentroids(random_state=42),
    "Cluster Centroids",
)

In [30]:
# Undersample training data (Random Undersampler)
rus = RandomUnderSampler(sampling_strategy=1.0, random_state=42)  # 1:1 fraud:non-fraud
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)
print("\nUndersampled Train Shape:", X_train_rus.shape)
print("Fraud in Undersampled Train:", y_train_rus.sum())


Undersampled Train Shape: (14764, 83)
Fraud in Undersampled Train: 7382


In [35]:
# tree based Models
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        use_label_encoder=False, eval_metric="logloss", random_state=42, n_jobs=-1
    ),
}

In [39]:
# Train and evaluate
results = {}
roc_curves = {}
threshold = 0.6
for name, model in models.items():
    model.fit(X_train_rus, y_train_rus)
    # y_pred = model.predict(X_test)
    y_pred = (y_pred_proba > threshold).astype(int)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # save the model
    dump(model, f"{name}.joblib")

    # Store results
    print(f"\n{name} - Classification Report:")
    print(classification_report(y_test, y_pred))
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"{name} ROC-AUC: {roc_auc:.4f}")
    results[name] = {
        "report": classification_report(y_test, y_pred, output_dict=True),
        "roc_auc": roc_auc,
    }


Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99   1473130
           1       0.04      0.97      0.08      1846

    accuracy                           0.97   1474976
   macro avg       0.52      0.97      0.53   1474976
weighted avg       1.00      0.97      0.98   1474976

Random Forest ROC-AUC: 0.9913


/opt/conda/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:51:15] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost - Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98   1473130
           1       0.04      0.95      0.07      1846

    accuracy                           0.97   1474976
   macro avg       0.52      0.96      0.53   1474976
weighted avg       1.00      0.97      0.98   1474976

XGBoost ROC-AUC: 0.9949


In [46]:
# try with SMOTE
smote = SMOTE(sampling_strategy=1, random_state=42)  # 1:1 fraud:non-fraud
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("\nSMOTE Train Shape:", X_train_smote.shape)
print("Fraud in SMOTE Train:", y_train_smote.sum())

# Models with SMOTE
models = {
    "Random Forest (SMOTE)": RandomForestClassifier(
        n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
    "XGBoost (SMOTE)": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        max_depth=10,
        scale_pos_weight=10,
        random_state=42,
        n_jobs=-1,
    ),
}

# Train and evaluate with threshold tuning
results = {}
roc_curves = {}
threshold = 0.0  # Higher for precision
for name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba > threshold).astype(int)

    # save the model
    dump(model, f"{name}.joblib")

    # Store results
    print(f"\n{name} - Classification Report (Threshold={threshold}):")
    print(classification_report(y_test, y_pred))
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"{name} ROC-AUC: {roc_auc:.4f}")
    results[name] = {
        "report": classification_report(y_test, y_pred, output_dict=True),
        "roc_auc": roc_auc,
    }


SMOTE Train Shape: (11785042, 83)
Fraud in SMOTE Train: 5892521

Random Forest (SMOTE) - Classification Report (Threshold=0.6):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1473130
           1       0.94      0.37      0.53      1846

    accuracy                           1.00   1474976
   macro avg       0.97      0.69      0.77   1474976
weighted avg       1.00      1.00      1.00   1474976

Random Forest (SMOTE) ROC-AUC: 0.9928


/opt/conda/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [18:28:16] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost (SMOTE) - Classification Report (Threshold=0.6):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1473130
           1       0.78      0.79      0.78      1846

    accuracy                           1.00   1474976
   macro avg       0.89      0.89      0.89   1474976
weighted avg       1.00      1.00      1.00   1474976

XGBoost (SMOTE) ROC-AUC: 0.9974


In [47]:
# try with SMOTE
smote = SMOTE(sampling_strategy=1, random_state=42)  # 1:1 fraud:non-fraud
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("\nSMOTE Train Shape:", X_train_smote.shape)
print("Fraud in SMOTE Train:", y_train_smote.sum())

# Models with SMOTE
models = {
    "Random Forest (SMOTE)": RandomForestClassifier(
        n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
    "XGBoost (SMOTE)": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        max_depth=12,
        scale_pos_weight=10,
        random_state=42,
        n_jobs=-1,
    ),
}

# Train and evaluate with threshold tuning
results = {}
roc_curves = {}
threshold = 0.5  # Higher for precision
for name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba > threshold).astype(int)

    # save the model
    dump(model, f"{name}1.joblib")

    # Store results
    print(f"\n{name} - Classification Report (Threshold={threshold}):")
    print(classification_report(y_test, y_pred))
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"{name} ROC-AUC: {roc_auc:.4f}")
    results[name] = {
        "report": classification_report(y_test, y_pred, output_dict=True),
        "roc_auc": roc_auc,
    }


SMOTE Train Shape: (11785042, 83)
Fraud in SMOTE Train: 5892521

Random Forest (SMOTE) - Classification Report (Threshold=0.5):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1473130
           1       0.85      0.48      0.61      1846

    accuracy                           1.00   1474976
   macro avg       0.92      0.74      0.81   1474976
weighted avg       1.00      1.00      1.00   1474976

Random Forest (SMOTE) ROC-AUC: 0.9928


/opt/conda/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [19:03:50] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost (SMOTE) - Classification Report (Threshold=0.5):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1473130
           1       0.86      0.77      0.81      1846

    accuracy                           1.00   1474976
   macro avg       0.93      0.89      0.91   1474976
weighted avg       1.00      1.00      1.00   1474976

XGBoost (SMOTE) ROC-AUC: 0.9977
